In [23]:
import pandas as pd

In [24]:
X_train = pd.read_csv("../data/X_train_data.csv")
y_train = pd.read_csv("../data/Y_train_data.csv")

In [25]:
X_train.columns

Index(['Unnamed: 0', 'Name', 'Genres', 'Type', 'Episodes', 'Aired',
       'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating',
       'Ranked', 'Popularity', 'Members', 'Favorites', 'Watching', 'Completed',
       'On-Hold', 'Dropped', 'Plan to Watch'],
      dtype='object')

## Data needs to be cleaned

Some columns will be dropped, and some will be transformed

### Columns that will remain unchanged:
 - Name
 - Type
 - Episodes
 - Producers
 - Licensors
 - Studios
 - Source
 - Rating
 - Ranked
 - Popularity
 - Members
 - Favorites
 - Watching
 - Completed
 - On-hold
 - Dropped
 - Plan to watch

### Columns to be transformed:
 - Genres (separating into separate columns)
 - Aired (extracting the year)
 - Duration (extracting the seconds)

### Columns that will be dropped:
 - Unnamed: 0
 - MAL_ID
 - English name (there is already a name variable)
 - Japanese name (there is already a name variable)
 - Premiered 
 - Score-10
 - Score-9
 - Score-8
 - Score-7
 - Score-6
 - Score-5
 - Score-4
 - Score-3
 - Score-2
 - Score-1

In [26]:
#cleaning the genres column
X_train['Genres'] = X_train['Genres'].astype(str).str.strip().str.split(r'\s*,\s*')
one_hot = X_train['Genres'].explode().str.get_dummies().groupby(level=0).sum()


In [27]:
X_train.drop("Genres", axis=1, inplace = True)
print("Old Genres column dropped")

Old Genres column dropped


In [28]:
X_train = X_train.join(one_hot)

In [29]:
X_train.columns

Index(['Unnamed: 0', 'Name', 'Type', 'Episodes', 'Aired', 'Producers',
       'Licensors', 'Studios', 'Source', 'Duration', 'Rating', 'Ranked',
       'Popularity', 'Members', 'Favorites', 'Watching', 'Completed',
       'On-Hold', 'Dropped', 'Plan to Watch', 'Action', 'Adventure', 'Cars',
       'Comedy', 'Dementia', 'Demons', 'Drama', 'Ecchi', 'Fantasy', 'Game',
       'Harem', 'Hentai', 'Historical', 'Horror', 'Josei', 'Kids', 'Magic',
       'Martial Arts', 'Mecha', 'Military', 'Music', 'Mystery', 'Parody',
       'Police', 'Psychological', 'Romance', 'Samurai', 'School', 'Sci-Fi',
       'Seinen', 'Shoujo', 'Shoujo Ai', 'Shounen', 'Shounen Ai',
       'Slice of Life', 'Space', 'Sports', 'Super Power', 'Supernatural',
       'Thriller', 'Unknown', 'Vampire', 'Yaoi', 'Yuri'],
      dtype='object')

In [30]:
#Dealing with the aired column now
X_train["Aired"].unique()

array(['Oct 21, 1994', 'Oct 11, 1999', 'Sep 13, 2003', ...,
       'Oct 5, 2010 to Dec 28, 2010', 'Apr 14, 2006 to Jun 23, 2006',
       'Apr 20, 2019'], shape=(10013,), dtype=object)

In [31]:
X_train['Year'] = X_train['Aired'].str.extract(r'(\d{4})').astype('float').astype('Int64')

In [32]:
X_train.Year.isna().sum()

np.int64(252)

In [33]:
X_train.Year

0        1994
1        1999
2        2003
3        2014
4        2003
         ... 
14044    2016
14045    2008
14046    2010
14047    2006
14048    2019
Name: Year, Length: 14049, dtype: Int64

In [34]:
X_train.drop("Aired", axis = 1, inplace = True)
X_train.columns

Index(['Unnamed: 0', 'Name', 'Type', 'Episodes', 'Producers', 'Licensors',
       'Studios', 'Source', 'Duration', 'Rating', 'Ranked', 'Popularity',
       'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped',
       'Plan to Watch', 'Action', 'Adventure', 'Cars', 'Comedy', 'Dementia',
       'Demons', 'Drama', 'Ecchi', 'Fantasy', 'Game', 'Harem', 'Hentai',
       'Historical', 'Horror', 'Josei', 'Kids', 'Magic', 'Martial Arts',
       'Mecha', 'Military', 'Music', 'Mystery', 'Parody', 'Police',
       'Psychological', 'Romance', 'Samurai', 'School', 'Sci-Fi', 'Seinen',
       'Shoujo', 'Shoujo Ai', 'Shounen', 'Shounen Ai', 'Slice of Life',
       'Space', 'Sports', 'Super Power', 'Supernatural', 'Thriller', 'Unknown',
       'Vampire', 'Yaoi', 'Yuri', 'Year'],
      dtype='object')

In [35]:
#dealing with the duration column
X_train.Duration.head(20)

0           28 min. per ep.
1                   54 min.
2                   20 min.
3           24 min. per ep.
4           30 min. per ep.
5           23 min. per ep.
6           30 min. per ep.
7                    5 min.
8                    5 min.
9                   52 min.
10          23 min. per ep.
11           1 min. per ep.
12                  58 min.
13                  55 min.
14                   1 min.
15          22 min. per ep.
16                   4 min.
17                  53 min.
18          23 min. per ep.
19    1 hr. 36 min. per ep.
Name: Duration, dtype: object

In [36]:
import re

def extract_seconds(s):
    s = str(s).lower()

    hr_match  = re.search(r'(\d+)\s*hr', s)
    min_match = re.search(r'(\d+)\s*min', s)
    sec_match = re.search(r'(\d+)\s*sec', s)

    hours = int(hr_match.group(1)) if hr_match else 0
    minutes = int(min_match.group(1)) if min_match else 0
    seconds = int(sec_match.group(1)) if sec_match else 0

    total_seconds = hours*3600 + minutes*60 + seconds

    return total_seconds if total_seconds > 0 else None

In [37]:
X_train["Seconds"] = X_train["Duration"].apply(extract_seconds)

In [38]:
X_train.Seconds

0        1680.0
1        3240.0
2        1200.0
3        1440.0
4        1800.0
          ...  
14044    1200.0
14045      30.0
14046    1440.0
14047    1320.0
14048     240.0
Name: Seconds, Length: 14049, dtype: float64

In [39]:
X_train.drop("Duration", axis =1, inplace = True)
print("Dropped Duration")

Dropped Duration


In [43]:
X_train.drop("Unnamed: 0", axis=1, inplace = True)


In [44]:
X_train.columns

Index(['Name', 'Type', 'Episodes', 'Producers', 'Licensors', 'Studios',
       'Source', 'Rating', 'Ranked', 'Popularity', 'Members', 'Favorites',
       'Watching', 'Completed', 'On-Hold', 'Dropped', 'Plan to Watch',
       'Action', 'Adventure', 'Cars', 'Comedy', 'Dementia', 'Demons', 'Drama',
       'Ecchi', 'Fantasy', 'Game', 'Harem', 'Hentai', 'Historical', 'Horror',
       'Josei', 'Kids', 'Magic', 'Martial Arts', 'Mecha', 'Military', 'Music',
       'Mystery', 'Parody', 'Police', 'Psychological', 'Romance', 'Samurai',
       'School', 'Sci-Fi', 'Seinen', 'Shoujo', 'Shoujo Ai', 'Shounen',
       'Shounen Ai', 'Slice of Life', 'Space', 'Sports', 'Super Power',
       'Supernatural', 'Thriller', 'Unknown', 'Vampire', 'Yaoi', 'Yuri',
       'Year', 'Seconds'],
      dtype='object')

There are still some unknown values in type, episodes, producers, licensors, studios, source, rating, year, seconds. So far no rows have been dropped, but certain columns have. This will need to be altered with the test data. 

In [45]:

X_train.Seconds.unique()

array([1.68e+03, 3.24e+03, 1.20e+03, 1.44e+03, 1.80e+03, 1.38e+03,
       3.00e+02, 3.12e+03, 6.00e+01, 3.48e+03, 3.30e+03, 1.32e+03,
       2.40e+02, 3.18e+03, 5.76e+03, 5.40e+03, 7.20e+02, 1.50e+03,
       3.72e+03, 5.40e+02, 5.70e+03, 3.00e+03, 1.80e+02, 8.40e+02,
       1.56e+03, 3.60e+02, 9.00e+02, 6.00e+02, 2.70e+03, 1.20e+02,
       5.46e+03,      nan, 6.60e+02, 6.60e+03, 1.08e+03, 1.62e+03,
       3.10e+01, 6.66e+03, 9.60e+02, 2.00e+01, 4.80e+02, 2.82e+03,
       4.20e+02, 4.50e+01, 7.80e+02, 2.16e+03, 2.40e+03, 1.02e+03,
       4.02e+03, 4.80e+03, 2.10e+03, 4.50e+03, 4.20e+03, 5.58e+03,
       6.24e+03, 3.06e+03, 2.22e+03, 2.90e+01, 3.00e+01, 6.72e+03,
       1.50e+01, 3.60e+03, 3.54e+03, 6.30e+03, 5.22e+03, 5.10e+03,
       6.00e+03, 1.86e+03, 5.10e+01, 3.96e+03, 5.34e+03, 1.74e+03,
       7.56e+03, 4.44e+03, 3.66e+03, 5.88e+03, 2.80e+01, 2.34e+03,
       2.64e+03, 2.88e+03, 4.86e+03, 7.08e+03, 1.92e+03, 4.20e+01,
       3.90e+01, 7.14e+03, 5.16e+03, 8.10e+03, 2.52e+03, 1.14e

In [42]:
X_train.to_csv("Cleaned_X_train.csv")